# Barbados R.O.A.D. — **class-wise models**: one separate LoRA + one specialised prompt per class

This notebook trains **a separate model for every data class**. Each is a brand-new LoRA adapter on the base Qwen-VL model, trained **only on that class's lines**, with a **prompt written for that class**. Every validation and test line is decoded by the model of its own class.

### Classes (from the image pixels only, so they work identically on test)
| Key | Rule | Train lines | Documents / conventions (measured on Train.csv) |
|---|---|---:|---|
| **B** | image height > 150 px | 1,043 | 1670s–1710s, high-res scans; `^` in 25 % of lines, `:` suspensions in 17 %, ff- words, spellings `heyres`, `saide` |
| **A** | height ≤ 150 px | 3,055 | 1630s–1660s; `&` in 14 %, ye/yt in 11 %, `sd` for said, `heires`, `assignes` |
| A1 | A and height ≤ 58 px | 1,669 | tight crops; heavy `&` and ye/yt |
| A2 | A, taller, parchment grey ≤ 194 | 383 | dark parchment, larger hand; `&` and ye rare; short lines |
| A3 | A, taller, parchment grey > 194 | 1,003 | loose crops showing neighbouring lines; longest lines |

`CLASS_SCHEME = "class"` trains 2 models (A, B). `CLASS_SCHEME = "subclass"` trains 4 models (A1, A2, A3, B).

### What happens
1. **Train.** For each base model in `MODEL_PATHS` and each class, a fresh LoRA is trained on that class's training lines with that class's prompt, using realistic augmentation. Optional `CLASS_REPLAY` mixes in a share of other-class lines; the default 0 means fully separate. The adapters are saved as `final/<model>/fold0/class_<key>/`.
2. **Decode.** Every validation, test and unlisted line is decoded by its class's adapter, with every decoding variant. Scores are printed overall, per class and per group.
3. **Compare (optional).** `BASELINE_PRED_CSV` can point to a *shared-model* validation prediction file from `barbados-2-enhanced.ipynb`. Every trial then shows the paired difference, and the forensics cell shows it per class.
4. **Combine and write.** With several base models, their class-routed candidates are combined by MBR. The best validated configuration writes `submission.csv`; `pseudo_labels.csv` is also written.

> **Evidence note** (`reports/Per class specialist HTR models.md`): in every published comparison, models trained on one domain only lost to one shared model, most on the smallest domain. This notebook lets you measure that on R.O.A.D. directly. Compare it with a shared-model run before submitting.

In [ ]:
# =========================================================
# Cell 1 — OFFLINE install from the wheels dataset
# =========================================================
import os, sys, subprocess, glob

os.environ["HF_HUB_OFFLINE"]         = "1"
os.environ["TRANSFORMERS_OFFLINE"]   = "1"
os.environ["HF_DATASETS_OFFLINE"]    = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import glob as _glob
import os as _os
WHEELS = "/kaggle/input/datasets/haniagamal/teleco-wheels/wheels"
if not _glob.glob(_os.path.join(WHEELS, "*.whl")):
    _hits = _glob.glob("/kaggle/input/**/*.whl", recursive=True)
    if _hits:
        WHEELS = _os.path.dirname(sorted(_hits, key=len)[0])
print("wheels dir:", WHEELS, "  found:",
      len(_glob.glob(_os.path.join(WHEELS, "*.whl"))), "wheels")

In [ ]:
%%capture
!pip uninstall -y torchao
!pip install --no-index --find-links {WHEELS} \
    transformers datasets peft trl jiwer timm librosa soundfile \
    accelerate bitsandbytes huggingface_hub torchao==0.16
print("Dependencies installed (offline).")

In [ ]:
# =========================================================
# Cell 2 — imports
# =========================================================
import ast, glob, json, shutil, gc, time, math, re, copy, collections
import torch
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
import numpy as np
import pandas as pd
from PIL import Image
from glob import glob as _glob
from tqdm.auto import tqdm
from sklearn.model_selection import KFold

import transformers
from datasets import Dataset
from transformers import AutoProcessor, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, PeftModel

try:
    from rapidfuzz.distance import Levenshtein as _RFLev      # ships with jiwer>=3
    def _lev(a, b):
        return _RFLev.distance(a, b)
except Exception:                                              # pure-python fallback
    def _lev(a, b):
        n, m = len(a), len(b)
        if n == 0: return m
        if m == 0: return n
        prev = list(range(m + 1))
        for i in range(1, n + 1):
            cur = [i] + [0] * m; ai = a[i - 1]
            for j in range(1, m + 1):
                cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ai != b[j - 1]))
            prev = cur
        return prev[m]

def _auto_vlm_class():
    for name in ("AutoModelForImageTextToText", "AutoModelForVision2Seq"):
        if hasattr(transformers, name):
            return getattr(transformers, name)
    raise ImportError("no AutoModelForImageTextToText / AutoModelForVision2Seq")
AutoVLM = _auto_vlm_class()

print("transformers", transformers.__version__, "| VLM class:", AutoVLM.__name__)
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "| bf16", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
SESSION_T0 = time.time()

In [ ]:
# =========================================================
# Cell 3 — CFG for CLASS-WISE models
# =========================================================
TRIAL_ID          = "CLS_001"
TRIAL_DESCRIPTION = ("Separate LoRA per class, fresh from the base model, trained on that class only, "
                     "with a class-specific prompt; class-routed decoding; full Fold-0 val")

COMMON_PROMPT = ("Transcribe the handwritten line exactly as written. This is not modern English: keep the original "
                 "spelling, capital letters, abbreviations and marks (^ for superscript letters, : for suspensions, "
                 "&, ff). Do not modernize, expand or correct anything. If parts of neighbouring lines are visible, "
                 "transcribe only the central line.")
_A = (" This line comes from a Barbados legal record of the 1630s-1660s. Common conventions in these documents: "
      "& for 'and', ye and yt for 'the' and 'that', sd for 'said', and spellings such as heires, assignes, publique.")
_B = (" This line comes from a Barbados legal record of the 1670s-1710s, scanned at high resolution. Common "
      "conventions in these documents: superscript abbreviations written with ^ (W^m, y^e, M^r, Adm^rs, Ex^rs), "
      "colon suspensions (Tho:, Exec:^rs, S:^d), ff at the start of words, and spellings such as heyres and saide.")

class CFG:
    WHEELS_DIR = WHEELS
    # ---- inputs ----
    MODEL_PATHS = [
        "/kaggle/input/models/qwen-lm/qwen-3-vl/transformers/8b-instruct/1",
    ]
    GRAD_CKPT_MODELS = ("32b",)
    DATA_DIR    = "/kaggle/input/datasets/haniagamal/road-barbados"
    TRAIN_CSV   = None
    TEST_CSV    = None
    IMAGES_DIR  = None
    AUTODISCOVER = True
    # ---- outputs ----
    OUTPUT_DIR     = "/kaggle/working/qwen-vl-ocr"
    FINAL_DIR      = "/kaggle/working/qwen-vl-ocr/final"     # -> FINAL_DIR/<model>/fold0/class_<key>/
    WORK_DIR       = "/kaggle/working"
    SUBMISSION_CSV = "/kaggle/working/submission.csv"
    KEEP_ADAPTERS  = True
    ADAPTERS_DIR   = None     # DO_TRAIN=False: folder holding <model>/fold0/class_<key>/ ; None -> search /kaggle/input
    PREV_RESULTS_DIRS = []

    # ---- protocol ----
    RUN_MODE        = "fold0"     # "fold0": train class models on Fold-0 train, score ALL 819 val lines | "full": all lines
    FOLD            = 0
    N_FOLDS         = 5
    CKPT_SELECTION  = "last"      # class models train on ALL their Fold-0 train lines (small classes -> no inner slice)
    INNER_CKPT_FRAC = 0.08
    TIME_BUDGET_H   = 11.0

    # ---- CLASS-WISE setup ----
    CLASS_SCHEME = "class"        # "class": models A, B | "subclass": models A1, A2, A3, B
    CLASS_PROMPTS = {             # one prompt per class model, used in training AND inference
        "A":  COMMON_PROMPT + _A,
        "B":  COMMON_PROMPT + _B,
        "A1": COMMON_PROMPT + _A + " The crop is tight around the line; '&' and ye/yt are very common.",
        "A2": COMMON_PROMPT + _A + " The parchment is dark and the hand is large; in these documents '&' and ye are rare and lines are short.",
        "A3": COMMON_PROMPT + _A + " The crop is loose and often shows parts of the lines above and below; the central line is usually long.",
    }
    CLASS_REPLAY = 0.0            # share of each class model's rows drawn from OTHER classes (0 = fully separate)
    CLASS_HP = {}                 # per-class overrides, e.g. {"A2": {"NUM_TRAIN_EPOCHS": 2}, "B": {"LEARNING_RATE": 1e-4}}
    BASELINE_PRED_CSV = None      # optional: shared-model val predictions (ID, pred), e.g. an enhanced-run
                                  #   trial_predictions/EXP_002__qwen_3_vl_transformers_8b_instruct_1__D0_orig__val.csv (measured 0.90947)

    # ---- training (per class model; overridable per model and per class) ----
    PER_DEVICE_TRAIN_BATCH_SIZE  = 4
    GRADIENT_ACCUMULATION_STEPS  = 1
    NUM_TRAIN_EPOCHS             = 1
    LEARNING_RATE                = 2e-4
    LR_SCHEDULER                 = "cosine"
    WARMUP_RATIO                 = 0.03
    DATALOADER_NUM_WORKERS       = 4
    MAX_PIXELS                   = 2_000_000
    MIN_PIXELS                   = 256*28*28
    ATTN_IMPL                    = None
    USE_BF16                     = True
    R                            = 32
    LORA_ALPHA                   = 64
    LORA_DROPOUT                 = 0.05
    LORA_VISION                  = True
    SEED                         = 42
    MODEL_OVERRIDES = {"32b": {"LEARNING_RATE": 1e-4}}

    # ---- class rule (pixels only) ----
    CLASS_HEIGHT_SPLIT  = 150
    GROUP_A1_MAX_HEIGHT = 58
    GROUP_A2_MAX_PAPER  = 194
    CLASS_HINT = True             # internal: pass each line's class to the prompt builder (required here)
    HINT_LEVEL = "group"
    CLASS_AUG_COPIES = None

    # ---- augmentation (realistic profile, see barbados-2-enhanced) ----
    USE_AUGMENT   = True
    AUG_COPIES    = 1
    AUG_PROFILE   = "realistic"
    AUG_RESAMPLE  = True
    AUG_STROKE_MODE = "both"
    AUG_ROTATE  = True;  AUG_ROTATE_P  = 0.5;  AUG_ROTATE_DEG    = 4.0
    AUG_SHEAR   = True;  AUG_SHEAR_P   = 0.4;  AUG_SHEAR_MAX     = 0.15
    AUG_ELASTIC = True;  AUG_ELASTIC_P = 0.4;  AUG_ELASTIC_ALPHA = 6.0;  AUG_ELASTIC_SIGMA = 8.0
    AUG_HJITTER = True;  AUG_HJITTER_P = 0.4;  AUG_HJITTER_FRAC  = 0.12
    AUG_PAD     = True;  AUG_PAD_P     = 0.3;  AUG_PAD_FRAC      = 0.06
    AUG_PHOTO   = True;  AUG_PHOTO_P   = 0.7;  AUG_BRIGHT = 0.20;  AUG_CONTRAST = 0.20;  AUG_GAMMA = 1.4
    AUG_TINT    = True;  AUG_TINT_P    = 0.5;  AUG_TINT_MAX = 0.12;  AUG_DESAT_MAX = 0.40
    AUG_ILLUM   = True;  AUG_ILLUM_P   = 0.4;  AUG_ILLUM_MAX = 0.35
    AUG_STROKE  = True;  AUG_STROKE_P  = 0.35
    AUG_BLEED   = True;  AUG_BLEED_P   = 0.25;  AUG_BLEED_ALPHA = 0.12
    AUG_NOISE   = True;  AUG_NOISE_P   = 0.5;  AUG_NOISE_STD = 0.1
    AUG_JPEG    = True;  AUG_JPEG_P    = 0.4;  AUG_JPEG_QMIN = 40;  AUG_JPEG_QMAX = 80

    CORRUPT_IDS = {
        "79tMUVyfIdy3GzkG",  "F8DYDDp2AvW9Dytw", "JU7lRwk3jKkus24Z", "VmrEALeZiP1Y6nF9", "t0UrASljcgzvBAnO",
    }

    # ---- decoding / ensembles / final run ----
    MAX_NEW_TOKENS = 256
    BATCH_SIZE     = 8
    DECODE_VARIANTS = {   # EXP_002 (GPU, 8B): D0 0.90947 vs D1 0.90902 (P(D1 better)=0.07) -> keep the penalty
        "D0_orig":  dict(num_beams=3, repetition_penalty=1.2, no_repeat_ngram_size=6, n_return=1),
        "D2_b5nb":  dict(num_beams=5, repetition_penalty=1.0, no_repeat_ngram_size=0, n_return=5),
    }
    BASELINE_VARIANT = "D0_orig"
    MBR_TEMPERATURE  = 1.0
    ENSEMBLE_METHODS = ("mbr_nbest", "mbr_top1")
    CHAMPION_DECODE   = "D2_b5nb"
    CHAMPION_ENSEMBLE = "mbr_nbest"   # RUN_MODE="full": across base models; with one model -> MBR over its 5-best
    # ---- pseudo-labels ----
    PREDICT_UNLISTED = True
    PSEUDO_LABEL_CSV = None
    PSEUDO_KEEP_FRAC = 0.7
    PSEUDO_PER_CLASS = True
    # ---- flow ----
    DO_TRAIN = True
    DO_INFER = True
    PREDICT_TEST = True
    OCR_PROMPT = COMMON_PROMPT        # fallback only (lines always carry their class)
    # unused by this notebook, kept so the shared helper cells run unchanged
    SPECIALISTS = {}; SPECIALIST_MODELS = (); CLASS_SAMPLING_T = None; CLASS_ROUTING = False; ROUTING_LEVEL = "class"
    LR_SCHEDULER_SPEC = None

TRIAL_CONFIG = {k: v for k, v in vars(CFG).items() if not k.startswith("_") and not callable(v)}
print("TRIAL:", TRIAL_ID, "|", TRIAL_DESCRIPTION)
print("CLASS_SCHEME:", CFG.CLASS_SCHEME, "| models:", len(CFG.MODEL_PATHS), "| replay:", CFG.CLASS_REPLAY)

In [ ]:
# =========================================================
# Cell 4 — resolve & sanity-check paths
# =========================================================
def _first_existing(paths):
    for p in paths:
        if p and os.path.exists(p):
            return p
    return None

def _find_under(root, filename):
    hits = _glob(os.path.join(root, "**", filename), recursive=True)
    return hits[0] if hits else None

def resolve_paths(cfg):
    train_csv = cfg.TRAIN_CSV or os.path.join(cfg.DATA_DIR, "Train.csv")
    test_csv  = cfg.TEST_CSV  or os.path.join(cfg.DATA_DIR, "Test.csv")
    images    = cfg.IMAGES_DIR or _first_existing([
        os.path.join(cfg.DATA_DIR, "images", "images"),
        os.path.join(cfg.DATA_DIR, "images"),
    ]) or os.path.join(cfg.DATA_DIR, "images", "images")
    if cfg.AUTODISCOVER:
        if not os.path.exists(train_csv):
            train_csv = _find_under("/kaggle/input", "Train.csv") or train_csv
        if not os.path.exists(test_csv):
            test_csv = _find_under("/kaggle/input", "Test.csv") or test_csv
        if not os.path.isdir(images) or not _glob(os.path.join(images, "*.jpg")):
            cand = _glob("/kaggle/input/**/*.jpg", recursive=True)
            if cand:
                images = os.path.dirname(cand[0])
    return train_csv, test_csv, images

TRAIN_CSV, TEST_CSV, IMAGES_DIR = resolve_paths(CFG)
TRIAL_PRED_DIR = os.path.join(CFG.WORK_DIR, "trial_predictions")
TRIAL_CFG_DIR  = os.path.join(CFG.WORK_DIR, "trial_configs")
TRIAL_RESULTS  = os.path.join(CFG.WORK_DIR, "trial_results.csv")
for _d in (CFG.OUTPUT_DIR, CFG.FINAL_DIR, TRIAL_PRED_DIR, TRIAL_CFG_DIR):
    os.makedirs(_d, exist_ok=True)

print("TRAIN_CSV :", TRAIN_CSV,  "->", os.path.exists(TRAIN_CSV))
print("TEST_CSV  :", TEST_CSV,   "->", os.path.exists(TEST_CSV))
print("IMAGES_DIR:", IMAGES_DIR, "->", os.path.isdir(IMAGES_DIR))
for _m in CFG.MODEL_PATHS:
    print("MODEL     :", _m, "->", os.path.isdir(_m))
assert os.path.exists(TRAIN_CSV), "Train.csv not found"
assert os.path.isdir(IMAGES_DIR), "images dir not found"
assert all(os.path.isdir(m) for m in CFG.MODEL_PATHS), "a model dir is missing"

In [ ]:
# =========================================================
# Cell 5 — text / image utils + augmentation stack
# =========================================================
def clean_label(x):
    return " ".join(str(x).replace("\n", " ").split()).strip()

def clean_output(text: str) -> str:
    text = str(text)
    for tag in ["assistant", "user", "<|assistant|>", "<|user|>"]:
        if tag in text:
            text = text.split(tag)[-1]
    return " ".join(text.split()).strip()

# identical resize ladder for train AND inference (downscale only; the processor then applies MIN/MAX_PIXELS)
def load_image(path):
    img = Image.open(path).convert("RGB")
    w, h = img.size
    aspect = w / h
    if   aspect > 10: img.thumbnail((2048, 384))
    elif aspect > 5:  img.thumbnail((2048, 512))
    elif aspect > 3:  img.thumbnail((1600, 640))
    else:             img.thumbnail((1536, 768))
    return img

# Augmentation magnitudes matched to the full-data forensics of all 6,159 images (FULL_DATA_AUDIT §4.2).
# measured (5th-95th pct)                        original barbados-2              realistic
REALISTIC_AUG = dict(
    AUG_ROTATE_DEG=1.0,  AUG_ROTATE_P=0.5,        # skew -1..+1 deg              | +-4 deg
    AUG_SHEAR_MAX=0.12,  AUG_SHEAR_P=0.4,         # slant spread -0.2..0.8       | 0.15
    AUG_ELASTIC_P=0.3,                            # hand wobble, kept mild       | 0.4
    AUG_HJITTER_FRAC=0.10,                        # x-height varies widely       | 0.12
    AUG_BRIGHT=0.15, AUG_CONTRAST=0.20, AUG_GAMMA=1.3, AUG_PHOTO_P=0.6,   # paper grey 145..216
    AUG_TINT_MAX=0.10, AUG_DESAT_MAX=0.30,        # paper tint R-B 40..71
    AUG_ILLUM_MAX=0.12, AUG_ILLUM_P=0.3,          # illumination CV <= 0.09      | 35 % vignette
    AUG_STROKE_P=0.15, AUG_STROKE_MODE="thicken", # strokes 1.9-3.8 px: 3x3 max-filter erases them
    AUG_BLEED_ALPHA=0.10, AUG_BLEED_P=0.25,       # mid-tone / show-through 3-20 %
    AUG_NOISE_STD=0.012, AUG_NOISE_P=0.3,         # sensor noise sigma 1.8-4.2   | 25.5 grey levels
    AUG_JPEG_QMIN=70, AUG_JPEG_QMAX=92, AUG_JPEG_P=0.3,   # all images q~85     | q40-80
    AUG_FILL="paper",                             # new pixels get the page's own parchment colour, not white
)

def apply_aug_profile(cfg):
    cfg.AUG_FILL = getattr(cfg, "AUG_FILL", "white")
    if getattr(cfg, "AUG_PROFILE", "original") == "realistic":
        for k, v in REALISTIC_AUG.items():
            setattr(cfg, k, v)
    print(f"[aug] profile={cfg.AUG_PROFILE} resample={cfg.AUG_RESAMPLE} rotate=±{cfg.AUG_ROTATE_DEG}° "
          f"noise σ={cfg.AUG_NOISE_STD*255:.1f} jpeg q{cfg.AUG_JPEG_QMIN}-{cfg.AUG_JPEG_QMAX} "
          f"stroke={cfg.AUG_STROKE_MODE}@{cfg.AUG_STROKE_P} illum={cfg.AUG_ILLUM_MAX} fill={cfg.AUG_FILL}")
apply_aug_profile(CFG)

def _paper_grey(gray):
    """mean grey of the pixels above the Otsu threshold (= the parchment), as in FULL_DATA_AUDIT §4."""
    hist = np.bincount(gray.ravel(), minlength=256).astype(np.float64)
    w = np.cumsum(hist); s = np.cumsum(np.arange(256) * hist); tot, sall = w[-1], s[-1]
    with np.errstate(divide="ignore", invalid="ignore"):
        between = (s * tot - sall * w) ** 2 / (w * (tot - w))
    t = int(np.nanargmax(np.where((w > 0) & (w < tot), between, -1)))
    bg = gray[gray > t]
    return float(bg.mean()) if bg.size else float(gray.mean())

_GROUP_CACHE = {}
def image_group(path):
    """Data group from the ORIGINAL pixels only (no labels) — FULL_DATA_AUDIT §6b:
       B  : height > 150 px            -> 1670s-1710s scans, ~77 px/char, 2x '^', 4x ':'
       A1 : height <= 58 px            -> tight crops, heavy '&' and 'ye'
       A2 : taller, parchment <= 194   -> dark parchment, large hand, almost no '&' / 'ye'
       A3 : taller, parchment  > 194   -> loose crops with neighbour lines, longest lines
    The rule reproduces the label-derived clusters with ~90 % agreement."""
    if path in _GROUP_CACHE:
        return _GROUP_CACHE[path]
    with Image.open(path) as im:
        h = im.size[1]
        if h > CFG.CLASS_HEIGHT_SPLIT:
            g = "B"
        elif h <= CFG.GROUP_A1_MAX_HEIGHT:
            g = "A1"
        else:
            g = "A2" if _paper_grey(np.asarray(im.convert("L"))) <= CFG.GROUP_A2_MAX_PAPER else "A3"
    _GROUP_CACHE[path] = g
    return g

def image_class(path):
    return image_group(path)[0]          # 'A' or 'B'

def key_match(grp, key):
    """does sub-class grp ('A1', ..., 'B') belong to specialist key ('A', 'B', 'A1', ...)?"""
    return grp == key or grp[0] == key

def ocr_prompt(grp=None):
    """CLASS_HINT: append the scan type so one shared model can switch spelling conventions per era."""
    if not (CFG.CLASS_HINT and grp):
        return CFG.OCR_PROMPT
    tag = grp if CFG.HINT_LEVEL == "group" else grp[0]
    return CFG.OCR_PROMPT + f" Scan type: {tag}."

def _paper_color(img):
    """median colour of the brighter half of the page = the parchment (measured background)."""
    a = np.asarray(img.convert("RGB"), np.float32).reshape(-1, 3)
    lum = a.mean(1)
    bg = a[lum >= np.median(lum)]
    return tuple(int(v) for v in np.median(bg, axis=0))

from PIL import ImageOps, ImageFilter
import io as _io
try:
    from scipy.ndimage import gaussian_filter as _gaussian_filter, map_coordinates as _map_coordinates
    _HAS_SCIPY = True
except Exception:
    _HAS_SCIPY = False

def _photo(arr, rng, b, c, g):
    a = arr / 255.0
    a = np.power(np.clip(a, 0, 1), float(np.exp(rng.uniform(-np.log(g), np.log(g)))))
    a = (a - 0.5) * (1.0 + float(rng.uniform(-c, c))) + 0.5
    a = a * (1.0 + float(rng.uniform(-b, b)))
    return np.clip(a, 0, 1) * 255.0

def _tint(arr, rng, tint_max, desat_max):
    a = (arr / 255.0).copy()
    gray = a.mean(axis=2, keepdims=True)
    d = float(rng.uniform(0, desat_max)); a = a * (1 - d) + gray * d
    a[..., 0] *= 1.0 + float(rng.uniform(0, tint_max))
    a[..., 2] *= 1.0 - float(rng.uniform(0, tint_max))
    return np.clip(a, 0, 1) * 255.0

def _illumination(arr, rng, imax):
    h, w = arr.shape[:2]
    cx, cy = float(rng.uniform(0, w)), float(rng.uniform(0, h))
    yy, xx = np.mgrid[0:h, 0:w]
    d = np.sqrt(((xx - cx) / max(w, 1)) ** 2 + ((yy - cy) / max(h, 1)) ** 2)
    field = (1.0 - float(imax) * (d / (d.max() + 1e-6))).astype(np.float32)[..., None]
    return np.clip(arr * field, 0, 255)

def _bleed(arr, rng, alpha):
    a = float(rng.uniform(0, alpha))
    return np.clip(arr * (1 - a) + arr[:, ::-1, :] * a, 0, 255)

def _elastic(arr, rng, alpha, sigma):
    h, w = arr.shape[:2]
    dx = _gaussian_filter(rng.rand(h, w).astype(np.float32) * 2 - 1, sigma) * alpha
    dy = _gaussian_filter(rng.rand(h, w).astype(np.float32) * 2 - 1, sigma) * alpha
    yy, xx = np.meshgrid(np.arange(h), np.arange(w), indexing="ij")
    coords = [np.clip(yy + dy, 0, h - 1), np.clip(xx + dx, 0, w - 1)]
    out = np.empty_like(arr)
    for ch in range(arr.shape[2]):
        out[..., ch] = _map_coordinates(arr[..., ch], coords, order=1, mode="reflect")
    return out

def augment_image(img, seed=None):
    """Label-preserving photometric + geometric stack, seeded per row (reproducible copies)."""
    if not getattr(CFG, "USE_AUGMENT", False):
        return img
    rng = np.random.RandomState(int(seed) & 0x7fffffff) if seed is not None else np.random
    fill = _paper_color(img) if CFG.AUG_FILL == "paper" else (255, 255, 255)
    if CFG.AUG_ROTATE and rng.rand() < CFG.AUG_ROTATE_P:
        img = img.rotate(float(rng.uniform(-CFG.AUG_ROTATE_DEG, CFG.AUG_ROTATE_DEG)),
                         resample=Image.BICUBIC, expand=True, fillcolor=fill)
    if CFG.AUG_SHEAR and rng.rand() < CFG.AUG_SHEAR_P:
        sh = float(rng.uniform(-CFG.AUG_SHEAR_MAX, CFG.AUG_SHEAR_MAX)); w, h = img.size
        xshift = abs(sh) * h
        img = img.transform((w + int(round(xshift)), h), Image.AFFINE,
                            (1, sh, -xshift if sh > 0 else 0.0, 0, 1, 0),
                            resample=Image.BICUBIC, fillcolor=fill)
    if CFG.AUG_STROKE and rng.rand() < CFG.AUG_STROKE_P:
        # dark ink on light paper: MinFilter = thicker strokes (ink spread), MaxFilter = thinner (can erase)
        thin = CFG.AUG_STROKE_MODE == "both" and rng.rand() >= 0.5
        img = img.filter(ImageFilter.MaxFilter(3) if thin else ImageFilter.MinFilter(3))
    if CFG.AUG_HJITTER and rng.rand() < CFG.AUG_HJITTER_P:
        w, h = img.size
        f = float(rng.uniform(1 - CFG.AUG_HJITTER_FRAC, 1 + CFG.AUG_HJITTER_FRAC))
        img = img.resize((w, max(1, int(round(h * f)))), Image.BICUBIC)
    if CFG.AUG_PAD and rng.rand() < CFG.AUG_PAD_P:
        w, h = img.size; pf = CFG.AUG_PAD_FRAC
        img = ImageOps.expand(img, (int(rng.uniform(0, pf) * w), int(rng.uniform(0, pf) * h),
                                    int(rng.uniform(0, pf) * w), int(rng.uniform(0, pf) * h)),
                              fill=fill)
    arr = np.asarray(img.convert("RGB"), np.float32)
    if CFG.AUG_ELASTIC and _HAS_SCIPY and rng.rand() < CFG.AUG_ELASTIC_P:
        arr = _elastic(arr, rng, CFG.AUG_ELASTIC_ALPHA, CFG.AUG_ELASTIC_SIGMA)
    if CFG.AUG_PHOTO and rng.rand() < CFG.AUG_PHOTO_P:
        arr = _photo(arr, rng, CFG.AUG_BRIGHT, CFG.AUG_CONTRAST, CFG.AUG_GAMMA)
    if CFG.AUG_TINT and rng.rand() < CFG.AUG_TINT_P:
        arr = _tint(arr, rng, CFG.AUG_TINT_MAX, CFG.AUG_DESAT_MAX)
    if CFG.AUG_ILLUM and rng.rand() < CFG.AUG_ILLUM_P:
        arr = _illumination(arr, rng, CFG.AUG_ILLUM_MAX)
    if CFG.AUG_BLEED and rng.rand() < CFG.AUG_BLEED_P:
        arr = _bleed(arr, rng, CFG.AUG_BLEED_ALPHA)
    if CFG.AUG_NOISE and rng.rand() < CFG.AUG_NOISE_P:
        arr = arr + rng.randn(*arr.shape).astype(np.float32) * (CFG.AUG_NOISE_STD * 255.0)
    out = Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))
    if CFG.AUG_JPEG and rng.rand() < CFG.AUG_JPEG_P:
        buf = _io.BytesIO()
        out.save(buf, format="JPEG", quality=int(rng.randint(CFG.AUG_JPEG_QMIN, CFG.AUG_JPEG_QMAX + 1)))
        buf.seek(0); out = Image.open(buf).convert("RGB")
    return out

def resolve_image(base_dir, image_id):
    exact = os.path.join(base_dir, f"{image_id}.jpg")
    if os.path.exists(exact):
        return exact
    matches = _glob(os.path.join(base_dir, f"{image_id}*.jpg"))
    return matches[0] if matches else None

def model_short_name(msrc):
    parts = [p for p in msrc.replace("\\", "/").strip("/").split("/") if p]
    return (re.sub(r"[^A-Za-z0-9]+", "_", "_".join(parts[-4:]))[:60] or "model")

def legacy_short_name(msrc):
    """adapter dir name written by the ORIGINAL barbados-2.ipynb (last 3 path parts)."""
    parts = [p for p in msrc.replace("\\", "/").strip("/").split("/") if p]
    return (re.sub(r"[^A-Za-z0-9]+", "_", "_".join(parts[-3:]))[:48] or "model")

def model_size_tag(name):
    m = re.search(r"(?<![0-9])(\d+b)(?![0-9])", name.lower())
    return m.group(1) if m else name[-12:]

def free_cuda(*names):
    for n in names:
        if n in globals():
            del globals()[n]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# =========================================================
# Cell 6 — model & processor loaders (pixel limits enforced + verified)
# =========================================================
def load_vlm_model(path, dtype, device_map=None):
    base = {"trust_remote_code": True}
    if device_map is not None:
        base["device_map"] = device_map
    if getattr(CFG, "ATTN_IMPL", None):
        base["attn_implementation"] = CFG.ATTN_IMPL
    for dkey in ("dtype", "torch_dtype"):
        try:
            return AutoVLM.from_pretrained(path, **{dkey: dtype}, **base)
        except TypeError:
            continue
        except Exception:
            if "attn_implementation" in base:
                base.pop("attn_implementation")
                try:
                    return AutoVLM.from_pretrained(path, **{dkey: dtype}, **base)
                except TypeError:
                    continue
            raise
    return AutoVLM.from_pretrained(path, **base)

def _repair_preprocessor(path):
    dst = os.path.join(CFG.WORK_DIR, "_patched_" + os.path.basename(path.rstrip("/\\")))
    if os.path.isdir(dst):
        shutil.rmtree(dst)
    def _ignore_weights(_d, names):
        return [n for n in names if n.endswith(
            (".safetensors", ".bin", ".pt", ".pth", ".gguf", ".h5", ".msgpack", ".onnx"))]
    shutil.copytree(path, dst, ignore=_ignore_weights)
    pcfg = os.path.join(dst, "preprocessor_config.json")
    cfg = {}
    if os.path.exists(pcfg):
        with open(pcfg, "r") as f:
            cfg = json.load(f)
    prefer = ["Qwen2VLImageProcessorFast", "Qwen2_5_VLImageProcessorFast", "Qwen3VLImageProcessorFast",
              "Qwen2VLImageProcessor", "Qwen2_5_VLImageProcessor", "Qwen3VLImageProcessor"]
    avail = [c for c in prefer if hasattr(transformers, c)]
    if avail:
        cfg["image_processor_type"] = avail[0]
    else:
        cfg.pop("image_processor_type", None)
    with open(pcfg, "w") as f:
        json.dump(cfg, f)
    return dst

def _smart_resize(h, w, factor, min_pixels, max_pixels):
    h_bar, w_bar = round(h / factor) * factor, round(w / factor) * factor
    if h_bar * w_bar > max_pixels:
        beta = math.sqrt((h * w) / max_pixels)
        h_bar = max(factor, math.floor(h / beta / factor) * factor)
        w_bar = max(factor, math.floor(w / beta / factor) * factor)
    elif h_bar * w_bar < min_pixels:
        beta = math.sqrt(min_pixels / (h * w))
        h_bar, w_bar = math.ceil(h * beta / factor) * factor, math.ceil(w * beta / factor) * factor
    return h_bar, w_bar

def _force_pixel_limits(ip, min_pixels, max_pixels):
    """transformers 5.0 honours ip.min_pixels/ip.max_pixels attributes; >=5.1x only reads ip.size.
    Set every representation so the limits hold on any version."""
    try:
        sz = dict(ip.size) if not isinstance(ip.size, dict) else dict(ip.size)
    except Exception:
        sz = {}
    sz = {k: v for k, v in sz.items() if v is not None}
    sz["shortest_edge"], sz["longest_edge"] = int(min_pixels), int(max_pixels)
    try:
        ip.size = sz
    except Exception:
        pass
    for k, v in (("min_pixels", min_pixels), ("max_pixels", max_pixels)):
        try:
            setattr(ip, k, int(v))
        except Exception:
            pass

def load_vlm_processor(path, max_pixels=None, min_pixels=None):
    kw_px = {}
    if max_pixels is not None and min_pixels is not None:
        kw_px = dict(min_pixels=int(min_pixels), max_pixels=int(max_pixels))
    attempts = [dict(use_fast=True), dict(use_fast=False),
                dict(use_fast=True, trust_remote_code=True), dict(use_fast=False, trust_remote_code=True)]
    proc, last = None, None
    for src in (path, None):
        if src is None:
            print("[processor] direct load failed; repairing. cause:", repr(last)[:200])
            src = _repair_preprocessor(path)
        for kw in attempts:
            for extra in (kw_px, {}):
                try:
                    proc = AutoProcessor.from_pretrained(src, **kw, **extra); break
                except Exception as e:
                    last = e
            if proc is not None: break
        if proc is not None: break
    if proc is None:
        raise last
    ip = proc.image_processor
    if kw_px:
        _force_pixel_limits(ip, min_pixels, max_pixels)
    # ---- verify on a probe strip: the grid must match smart_resize(min/max) ----
    probe = Image.new("RGB", (1060, 57), (220, 200, 170))
    fac = int(getattr(ip, "patch_size", 14)) * int(getattr(ip, "merge_size", 2))
    out = ip(images=[probe], return_tensors="pt")
    g = out["image_grid_thw"][0].tolist()
    got = (g[1] * ip.patch_size, g[2] * ip.patch_size)
    if kw_px:
        exp = _smart_resize(57, 1060, fac, min_pixels, max_pixels)
        assert tuple(got) == tuple(exp), f"pixel limits NOT applied: processor gives {got}, expected {exp}"
    print(f"[processor] {type(proc).__name__} | {type(ip).__name__} | probe 1060x57 -> {got[1]}x{got[0]} px, "
          f"{g[1]*g[2]//(ip.merge_size**2)} visual tokens")
    return proc

In [ ]:
# =========================================================
# Cell 7 — OFFICIAL metric, MBR / ROVER ensembling, error forensics, paired bootstrap
#   score = 0.5*(1 - word_edits_per_line/12) + 0.5*(1 - char_edits_per_line/55)   (higher = better)
# =========================================================
WORD_XMAX, CHAR_XMAX = 12.0, 55.0

def norm_text(s):
    if s is None or (isinstance(s, float) and math.isnan(s)):
        return ""
    return " ".join(str(s).split())

def line_edits(ref, hyp):
    r, h = norm_text(ref), norm_text(hyp)
    return _lev(r.split(), h.split()), _lev(r, h)

def line_cost(ref, hyp):
    w, c = line_edits(ref, hyp)
    return 0.5 * w / WORD_XMAX + 0.5 * c / CHAR_XMAX

def official_metric(refs, hyps):
    ed = np.array([line_edits(r, h) for r, h in zip(refs, hyps)], float).reshape(-1, 2)
    we, ce = ed[:, 0], ed[:, 1]
    nw = max(1, sum(len(norm_text(r).split()) for r in refs)); nc = max(1, sum(len(norm_text(r)) for r in refs))
    return {"score": float(0.5 * (1 - we.mean() / WORD_XMAX) + 0.5 * (1 - ce.mean() / CHAR_XMAX)),
            "word_edits_per_line": float(we.mean()), "char_edits_per_line": float(ce.mean()),
            "wer_corpus": float(we.sum() / nw), "cer_corpus": float(ce.sum() / nc),
            "n": len(refs), "empty": int(sum(not norm_text(h) for h in hyps))}

def mbr_select(cands, weights=None, return_risk=False):
    """Minimum-Bayes-risk pick under the official per-line cost. The minimum expected cost ('risk')
    doubles as a confidence: low risk = the candidate pool agrees."""
    cands = [norm_text(c) for c in cands]
    weights = [1.0] * len(cands) if weights is None else weights
    agg = collections.OrderedDict()
    for c, w in zip(cands, weights):
        agg[c] = agg.get(c, 0.0) + float(w)
    uniq = list(agg)
    if len(uniq) == 1:
        return (uniq[0], 0.0) if return_risk else uniq[0]
    wv = np.array([agg[u] for u in uniq]); wv = wv / wv.sum()
    risks = [sum(w * line_cost(c, h) for c, w in zip(uniq, wv)) for h in uniq]
    k = int(np.argmin(risks))
    return (uniq[k], float(risks[k])) if return_risk else uniq[k]

def nbest_posteriors(total_logprobs, temperature=1.0):
    s = np.asarray(total_logprobs, float) / max(temperature, 1e-6)
    s = s - s.max(); p = np.exp(s)
    return p / p.sum()

def _align(cons, hyp):
    n, m = len(cons), len(hyp)
    D = np.zeros((n + 1, m + 1), np.int32); D[:, 0] = np.arange(n + 1); D[0, :] = np.arange(m + 1)
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            D[i, j] = min(D[i-1, j-1] + (cons[i-1] != hyp[j-1]), D[i-1, j] + 1, D[i, j-1] + 1)
    i, j, out = n, m, []
    while i > 0 or j > 0:
        if i > 0 and j > 0 and D[i, j] == D[i-1, j-1] + (cons[i-1] != hyp[j-1]):
            out.append((i - 1, hyp[j - 1])); i -= 1; j -= 1
        elif i > 0 and D[i, j] == D[i-1, j] + 1:
            out.append((i - 1, None)); i -= 1
        else:
            out.append((None, hyp[j - 1])); j -= 1
    return out[::-1]

def rover(hyps, weights=None, level="word"):
    """ROVER: align every system's top-1 into a confusion network, weighted vote per slot
    (an absent system votes for deletion). level='word' matches the metric's 4.58x word premium."""
    split = (lambda s: norm_text(s).split()) if level == "word" else (lambda s: list(norm_text(s)))
    join = " ".join if level == "word" else "".join
    seqs = [split(h) for h in hyps]
    weights = [1.0] * len(seqs) if weights is None else list(weights)
    order = list(np.argsort(-np.asarray(weights), kind="stable"))
    seqs = [seqs[k] for k in order]; weights = [weights[k] for k in order]
    slots = [{0: t} for t in seqs[0]]
    for s in range(1, len(seqs)):
        cons = [collections.Counter(sl.values()).most_common(1)[0][0] if sl else "" for sl in slots]
        new = []
        for ci, t in _align(cons, seqs[s]):
            if ci is None:
                new.append({s: t})
            else:
                sl = dict(slots[ci])
                if t is not None:
                    sl[s] = t
                new.append(sl)
        slots = new
    out = []
    for sl in slots:
        agg = collections.Counter()
        for s, t in sl.items():
            agg[t] += weights[s]
        agg[None] += sum(weights[s] for s in range(len(seqs)) if s not in sl)
        best = max(agg.items(), key=lambda kv: kv[1])[0]
        if best is not None:
            out.append(best)
    return norm_text(join(out))

def paired_delta(refs, hyp_a, hyp_b, n_boot=10000, seed=0):
    """challenger B vs champion A — paired bootstrap over whole lines."""
    la = np.array([line_cost(r, h) for r, h in zip(refs, hyp_a)])
    lb = np.array([line_cost(r, h) for r, h in zip(refs, hyp_b)])
    d = la - lb
    rng = np.random.default_rng(seed)
    boots = d[rng.integers(0, len(d), (n_boot, len(d)))].mean(1)
    ea = np.array([line_edits(r, h) for r, h in zip(refs, hyp_a)]); eb = np.array([line_edits(r, h) for r, h in zip(refs, hyp_b)])
    return {"delta": float(d.mean()), "ci95_lo": float(np.percentile(boots, 2.5)), "ci95_hi": float(np.percentile(boots, 97.5)),
            "p_better": float((boots > 0).mean()), "improved": int((d > 1e-12).sum()), "worsened": int((d < -1e-12).sum()),
            "word_edits_saved": int(ea[:, 0].sum() - eb[:, 0].sum()), "char_edits_saved": int(ea[:, 1].sum() - eb[:, 1].sum())}

def _word_err_class(r, h):
    if r.lower() == h.lower(): return "casing"
    strip = lambda x: "".join(ch for ch in x if ch.isalnum())
    if strip(r) == strip(h): return "punct/markup"
    if strip(r).lower() == strip(h).lower(): return "casing+punct"
    d = _lev(r, h)
    return "1-char" if d == 1 else "2-char" if d == 2 else "3+char"

def error_forensics(refs, hyps, top=20):
    try:
        from rapidfuzz.distance import Levenshtein as L
    except Exception:
        return {"note": "rapidfuzz unavailable"}
    ops, csub, wsub, wcls = collections.Counter(), collections.Counter(), collections.Counter(), collections.Counter()
    cdel, cins, wdel, wins = collections.Counter(), collections.Counter(), collections.Counter(), collections.Counter()
    for r, h in zip(refs, hyps):
        r, h = norm_text(r), norm_text(h); rw, hw = r.split(), h.split()
        for op in L.editops(rw, hw):
            ops["word_" + op.tag] += 1
            if op.tag == "replace":
                wsub[f"{rw[op.src_pos]} -> {hw[op.dest_pos]}"] += 1; wcls[_word_err_class(rw[op.src_pos], hw[op.dest_pos])] += 1
            elif op.tag == "delete": wdel[rw[op.src_pos]] += 1
            else: wins[hw[op.dest_pos]] += 1
        for op in L.editops(r, h):
            ops["char_" + op.tag] += 1
            if op.tag == "replace": csub[f"{r[op.src_pos]!r}->{h[op.dest_pos]!r}"] += 1
            elif op.tag == "delete": cdel[repr(r[op.src_pos])] += 1
            else: cins[repr(h[op.dest_pos])] += 1
    return {"ops": dict(ops), "word_sub_classes": dict(wcls), "word_sub": wsub.most_common(top),
            "word_del": wdel.most_common(top), "word_ins": wins.most_common(top),
            "char_sub": csub.most_common(top), "char_del": cdel.most_common(top), "char_ins": cins.most_common(top)}

def score_by_group(refs, hyps, groups):
    g = np.asarray(groups); out = {}
    for k in sorted(set(g.tolist()), key=str):
        idx = np.where(g == k)[0]
        m = official_metric([refs[i] for i in idx], [hyps[i] for i in idx])
        out[str(k)] = f"n={len(idx):4d} score={m['score']:.4f} w/line={m['word_edits_per_line']:.3f} c/line={m['char_edits_per_line']:.3f}"
    return out

# sanity checks
assert line_edits("a b c", "") == (3, 5)
assert abs(official_metric(["a b c"], [""])["score"] - (0.5 * (1 - 3/12) + 0.5 * (1 - 5/55))) < 1e-12
assert rover(["the said John", "the sd John", "the said Jon"]) == "the said John"
assert mbr_select(["a b c", "a b c", "a x c"]) == "a b c"
print("official metric + ensemble helpers OK")

In [ ]:
# =========================================================
# Cell 8 — samples, Fold-0 split (identical IDs to barbados-2), repaired protocol
# =========================================================
def build_samples(df, base_dir):
    corrupt = set(CFG.CORRUPT_IDS)
    samples, missing, excluded = [], 0, 0
    for _, row in df.iterrows():
        image_id, label = str(row["ID"]).strip(), clean_label(row["Target"])
        if not image_id or not label:
            continue
        if image_id in corrupt:
            excluded += 1; continue
        path = resolve_image(base_dir, image_id)
        if path is None:
            missing += 1; continue
        samples.append({"id": image_id, "image": path, "text": label})
    print(f"[DATASET] found={len(samples)}, missing={missing}, excluded_corrupt={excluded}")
    return samples

df = pd.read_csv(TRAIN_CSV).sample(frac=1, random_state=CFG.SEED).reset_index(drop=True)
all_samples = build_samples(df, IMAGES_DIR)
FOLDS = list(KFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED).split(all_samples))

if CFG.RUN_MODE == "fold0":
    tr_idx, va_idx = FOLDS[CFG.FOLD]
else:                                   # "full": everything is training data, no validation
    tr_idx, va_idx = np.arange(len(all_samples)), np.array([], int)
_rng = np.random.RandomState(CFG.SEED + 1000 + CFG.FOLD)
_perm = _rng.permutation(tr_idx)
n_inner = int(round(CFG.INNER_CKPT_FRAC * len(_perm))) if CFG.CKPT_SELECTION == "inner_loss" else 0
inner_idx, core_idx = _perm[:n_inner], _perm[n_inner:]
core_samples  = [all_samples[i] for i in core_idx]
inner_samples = [all_samples[i] for i in inner_idx]
val_samples   = [all_samples[i] for i in va_idx]
VAL_IDS  = [s["id"] for s in val_samples]
VAL_GT   = [s["text"] for s in val_samples]
VAL_PATHS = [s["image"] for s in val_samples]
assert not (set(VAL_IDS) & {s["id"] for s in core_samples + inner_samples}), "validation leak!"
print(f"mode={CFG.RUN_MODE} fold={CFG.FOLD}: train={len(core_samples)} inner_ckpt={len(inner_samples)} "
      f"val={len(val_samples)} (ALL scored)")
# data group of every image (A1/A2/A3 = 1630s-60s scans, B = 1670s-1710s high-res scans) — from pixels only
for s in all_samples:
    s["grp"] = image_group(s["image"])
VAL_GROUP = [s["grp"] for s in val_samples]
VAL_CLASS = [g[0] for g in VAL_GROUP]
VAL_LEN_BUCKET = [("<50" if len(t) < 50 else "50-69" if len(t) < 70 else "70+") for t in VAL_GT]
_cc = lambda ss: dict(sorted(collections.Counter(s["grp"] for s in ss).items()))
print(f"group balance  train={_cc(core_samples)}  inner={_cc(inner_samples)}  val={_cc(val_samples)}")

_test_df = pd.read_csv(TEST_CSV, encoding="utf-8-sig")
_test_df.columns = [c.lstrip("\ufeff") for c in _test_df.columns]
TEST_IDS   = [str(r["ID"]).strip() for r in _test_df.to_dict("records")]
TEST_PATHS = [os.path.join(IMAGES_DIR, f"{rid}.jpg") for rid in TEST_IDS]
print("test rows:", len(TEST_IDS), "| missing test images:", sum(not os.path.exists(p) for p in TEST_PATHS))
_known = set(TEST_IDS) | {str(i).strip() for i in pd.read_csv(TRAIN_CSV).ID}
UNL_IDS = sorted(os.path.basename(p)[:-4] for p in _glob(os.path.join(IMAGES_DIR, "*.jpg"))
                 if os.path.basename(p)[:-4] not in _known)
UNL_PATHS = [os.path.join(IMAGES_DIR, f"{i}.jpg") for i in UNL_IDS]
print("unlisted images (in images/, in neither CSV):", len(UNL_IDS))
TEST_GROUP = [image_group(p) if os.path.exists(p) else "A1" for p in TEST_PATHS]
UNL_GROUP  = [image_group(p) for p in UNL_PATHS]
TEST_CLASS = [g[0] for g in TEST_GROUP]
print("group balance  test=", dict(sorted(collections.Counter(TEST_GROUP).items())),
      " unlisted=", dict(sorted(collections.Counter(UNL_GROUP).items())))

# ---- optional pseudo-labels (test + unlisted) from a previous session's ensemble ----
pseudo_samples = []
if CFG.PSEUDO_LABEL_CSV and os.path.exists(CFG.PSEUDO_LABEL_CSV):
    _pl = pd.read_csv(CFG.PSEUDO_LABEL_CSV)
    _pl = _pl[_pl.Target.fillna("").str.split().str.len().between(3, 30)]        # drop empty / runaway lines
    _pl = _pl[_pl.ID.map(lambda i: os.path.exists(os.path.join(IMAGES_DIR, f"{i}.jpg")))]
    _pl["grp"] = [image_group(os.path.join(IMAGES_DIR, f"{i}.jpg")) for i in _pl.ID]
    if "keep_for_training" in _pl:  # file from work/08_build_pseudo_labels.py: calibrated per-class selection already made
        _pl = _pl[_pl.keep_for_training.astype(str).str.lower().isin(["true", "1"])]
    elif CFG.PSEUDO_PER_CLASS:      # class B is harder -> higher risk; a global cut would drop most of it
        _q = _pl.groupby(_pl.grp.str[0]).risk.transform(lambda r: r.quantile(CFG.PSEUDO_KEEP_FRAC))
        _pl = _pl[_pl.risk <= _q]
    else:
        _pl = _pl[_pl.risk <= _pl.risk.quantile(CFG.PSEUDO_KEEP_FRAC)]
    pseudo_samples = [{"id": r.ID, "image": os.path.join(IMAGES_DIR, f"{r.ID}.jpg"), "text": clean_label(r.Target),
                       "grp": r.grp} for r in _pl.itertuples()]
    assert not ({s["id"] for s in pseudo_samples} & set(VAL_IDS)), "pseudo-labels must not cover validation lines"
    print(f"pseudo-labelled lines added to TRAIN: {len(pseudo_samples)} (risk <= q{CFG.PSEUDO_KEEP_FRAC:.2f}"
          f"{' per class' if CFG.PSEUDO_PER_CLASS else ''}) by group:", dict(collections.Counter(_pl.grp)))
pd.DataFrame({"ID": VAL_IDS}).to_csv(os.path.join(CFG.WORK_DIR, f"fold{CFG.FOLD}_val_ids.csv"), index=False)

In [ ]:
# =========================================================
# Cell 9 — collator with assistant-boundary label masking (masking unchanged from barbados-2)
# =========================================================
MASK_FAIL_COUNT = 0
_AUG_VISITS = collections.Counter()

def _aug_seed(row_seed):
    """AUG_RESAMPLE: a new, still reproducible, augmentation every time a line is seen.
    Mixes the per-process visit count (num_workers=0) with the DataLoader worker seed, which
    PyTorch re-draws every epoch from the Trainer-seeded generator (num_workers>0)."""
    if not CFG.AUG_RESAMPLE:
        return row_seed
    v = _AUG_VISITS[row_seed]; _AUG_VISITS[row_seed] += 1
    return (int(row_seed) * 1000003 + v * 7919 + int(torch.initial_seed()) % 1000000007) & 0x7fffffff

def collate_fn(examples, processor):
    global MASK_FAIL_COUNT
    images, texts, classes = [], [], []
    for ex in examples:
        if os.path.exists(ex["image"]):
            img = load_image(ex["image"])
            if CFG.USE_AUGMENT and ex.get("aug", False):
                img = augment_image(img, _aug_seed(ex.get("seed")))
            images.append(img); texts.append(ex["text"]); classes.append(ex.get("grp"))
    if len(images) == 0:
        return None
    tokenizer = processor.tokenizer
    full_msgs, prompt_msgs = [], []
    for img, label, cls in zip(images, texts, classes):
        user_turn = {"role": "user", "content": [{"type": "text", "text": ocr_prompt(cls)},
                                                 {"type": "image", "image": img}]}
        full_msgs.append([user_turn, {"role": "assistant", "content": [{"type": "text", "text": label}]}])
        prompt_msgs.append([user_turn])
    texts_out = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in full_msgs]
    batch = processor(text=texts_out, images=images, padding=True, truncation=True, return_tensors="pt")
    labels = batch["input_ids"].clone()
    for i, pm in enumerate(prompt_msgs):
        try:
            prompt_txt = processor.apply_chat_template(pm, tokenize=False, add_generation_prompt=True)
            prompt_ids = processor(text=[prompt_txt], images=[images[i]], return_tensors="pt", truncation=True)["input_ids"][0]
            n_prompt = int((prompt_ids != tokenizer.pad_token_id).sum().item())
            labels[i, :n_prompt] = -100
        except Exception:
            MASK_FAIL_COUNT += 1
    labels[labels == tokenizer.pad_token_id] = -100
    image_token = getattr(processor, "image_token", None) or "<|image_pad|>"
    image_token_id = tokenizer.convert_tokens_to_ids(image_token)
    if isinstance(image_token_id, int) and image_token_id >= 0:
        labels[labels == image_token_id] = -100
    batch["labels"] = labels
    return batch

In [ ]:
# =========================================================
# Cell 10 — batched N-best generation with sequence scores
#   returns, per image, a list of (text, total_logprob, n_tokens, truncated) sorted best-first
# =========================================================
def _gen_config(model, var):
    gc_ = copy.deepcopy(model.generation_config)
    gc_.do_sample = False
    for k in ("temperature", "top_p", "top_k"):
        try: setattr(gc_, k, None)
        except Exception: pass
    nb = int(var.get("num_beams", 1)); nr = min(int(var.get("n_return", 1)), nb)
    gc_.num_beams = nb
    gc_.num_return_sequences = nr
    gc_.max_new_tokens = CFG.MAX_NEW_TOKENS
    gc_.repetition_penalty = float(var.get("repetition_penalty", 1.0))
    gc_.no_repeat_ngram_size = int(var.get("no_repeat_ngram_size", 0))
    gc_.length_penalty = float(var.get("length_penalty", 1.0))
    gc_.early_stopping = True if nb > 1 else False
    gc_.output_scores = True
    gc_.return_dict_in_generate = True
    return gc_, nb, nr

def generate_nbest(model, processor, images, var, classes=None):
    classes = classes or [None] * len(images)
    prompts = [processor.apply_chat_template(
        [{"role": "user", "content": [{"type": "text", "text": ocr_prompt(c)}, {"type": "image", "image": im}]}],
        tokenize=False, add_generation_prompt=True) for im, c in zip(images, classes)]
    tok = processor.tokenizer
    old_side = tok.padding_side; tok.padding_side = "left"
    try:
        inputs = processor(text=prompts, images=images, return_tensors="pt", padding=True)
    finally:
        tok.padding_side = old_side
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    gcfg, nb, nr = _gen_config(model, var)
    with torch.inference_mode():
        out = model.generate(**inputs, generation_config=gcfg)
    P = inputs["input_ids"].shape[1]
    gen = out.sequences[:, P:]
    stop_ids = {tok.pad_token_id, tok.eos_token_id, tok.convert_tokens_to_ids("<|im_end|>")}
    stop_ids = {s for s in stop_ids if isinstance(s, int) and s >= 0}
    is_stop = torch.zeros_like(gen, dtype=torch.bool)
    for s in stop_ids:
        is_stop |= gen == s
    has_stop = is_stop.any(1)
    first_stop = torch.where(has_stop, is_stop.float().argmax(1), torch.full_like(has_stop, gen.shape[1], dtype=torch.long))
    n_tok = (first_stop + has_stop.long()).clamp(min=1)            # generated tokens incl. EOS
    if nb > 1 and getattr(out, "sequences_scores", None) is not None:
        lp = float(gcfg.length_penalty)
        total = out.sequences_scores.float() * n_tok.float() ** lp  # undo length normalisation
    else:
        ts = model.compute_transition_scores(out.sequences, out.scores, normalize_logits=True)
        valid = torch.arange(ts.shape[1], device=ts.device)[None, :] < n_tok[:, None].to(ts.device)
        total = torch.where(valid & torch.isfinite(ts), ts, torch.zeros_like(ts)).sum(1)
    texts = [clean_output(t) for t in processor.batch_decode(gen, skip_special_tokens=True)]
    res = []
    for b in range(len(images)):
        rows = []
        for r in range(nr):
            k = b * nr + r
            rows.append((texts[k], float(total[k]), int(n_tok[k]), bool(not has_stop[k])))
        res.append(rows)
    return res

def transcribe(model, processor, paths, var, desc=None):
    """-> list (aligned with paths) of N-best lists. Missing images -> [("", -inf, 0, False)]."""
    bs = max(1, int(CFG.BATCH_SIZE) // max(1, int(var.get("num_beams", 1)) // 3))
    out = [[("", float("-inf"), 0, False)] for _ in paths]
    idxs = [i for i, p in enumerate(paths) if p and os.path.exists(p)]
    for b in tqdm(range(0, len(idxs), bs), desc=desc, leave=False):
        chunk = idxs[b:b + bs]
        imgs = [load_image(paths[i]) for i in chunk]
        cls = [image_group(paths[i]) for i in chunk] if CFG.CLASS_HINT else None
        try:
            res = generate_nbest(model, processor, imgs, var, cls)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            res = [generate_nbest(model, processor, [im], var, [c] if cls else None)[0]
                   for im, c in zip(imgs, cls or [None] * len(imgs))]
        for i, r in zip(chunk, res):
            out[i] = r
    return out

In [ ]:
# =========================================================
# Cell 11 — trial harness: persistence, champion tracking, standard report block
# =========================================================
def _prev_results():
    frames = []
    for d in [CFG.WORK_DIR] + list(CFG.PREV_RESULTS_DIRS):
        p = os.path.join(d, "trial_results.csv")
        if os.path.exists(p):
            try: frames.append(pd.read_csv(p))
            except Exception: pass
    return pd.concat(frames, ignore_index=True).drop_duplicates("trial") if frames else pd.DataFrame()

def champion_row():
    r = _prev_results()
    if len(r) == 0 or "score" not in r: return None
    r = r[r.status.isin(["PROMOTED", "BASELINE", "EVALUATED"]) & r.score.notna()]
    return None if len(r) == 0 else r.sort_values("score", ascending=False).iloc[0]

def save_predictions(trial, ids, gts, nbests, chosen=None, split="val"):
    rows = []
    for k, i in enumerate(ids):
        nb = nbests[k]
        top = chosen[k] if chosen is not None else nb[0][0]
        rows.append({"ID": i, "GroundTruth": gts[k] if gts is not None else None, "pred": top,
                     "logprob": nb[0][1] if nb else None, "n_tokens": nb[0][2] if nb else None,
                     "truncated": nb[0][3] if nb else None,
                     "nbest": json.dumps([[t, round(s, 4)] for t, s, _, _ in nb], ensure_ascii=False)})
    p = os.path.join(TRIAL_PRED_DIR, f"{trial}__{split}.csv")
    pd.DataFrame(rows).to_csv(p, index=False)
    return p

def report_trial(trial, desc, model, variant, hyps, runtime_s, n_train, trunc=0, baseline_hyps=None, extra=None):
    m = official_metric(VAL_GT, hyps)
    base = paired_delta(VAL_GT, baseline_hyps, hyps) if baseline_hyps is not None else None
    ch = champion_row()
    ch_score = float(ch.score) if ch is not None else float("nan")
    peak = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
    print("=" * 66)
    print(f"TRIAL: {trial}\nDESCRIPTION: {desc}\nMODEL: {model} | DECODE: {variant}")
    print("=" * 66)
    print(f"Official validation score : {m['score']:.5f}")
    print(f"Word edits / line (LB WER) : {m['word_edits_per_line']:.4f}   corpus WER {m['wer_corpus']:.4f}")
    print(f"Char edits / line (LB CER) : {m['char_edits_per_line']:.4f}   corpus CER {m['cer_corpus']:.4f}")
    if base is not None:
        print(f"Delta vs baseline          : {base['delta']:+.5f}  CI95 [{base['ci95_lo']:+.5f}, {base['ci95_hi']:+.5f}]"
              f"  P(better)={base['p_better']:.3f}  improved={base['improved']} worsened={base['worsened']}"
              f"  word/char edits saved={base['word_edits_saved']}/{base['char_edits_saved']}")
    print(f"Delta vs champion          : {m['score'] - ch_score:+.5f}  (champion={ch_score:.5f})")
    print(f"Runtime                    : {runtime_s/60:.1f} min   Peak GPU RAM : {peak:.1f} GB")
    print(f"Train examples             : {n_train}   Validation examples : {m['n']}")
    print(f"Empty predictions          : {m['empty']}   Truncated predictions : {trunc}")
    print("By class (A<=150px, B>150px):", score_by_group(VAL_GT, hyps, VAL_CLASS))
    print("By group (A1/A2/A3/B)      :", score_by_group(VAL_GT, hyps, VAL_GROUP))
    print("By label length          :", score_by_group(VAL_GT, hyps, VAL_LEN_BUCKET))
    print("=" * 66)
    row = {"trial": trial, "session": TRIAL_ID, "description": desc, "model": model, "variant": variant,
           "score": m["score"], "word_edits_per_line": m["word_edits_per_line"], "char_edits_per_line": m["char_edits_per_line"],
           "wer_corpus": m["wer_corpus"], "cer_corpus": m["cer_corpus"],
           "delta_baseline": base["delta"] if base else np.nan, "delta_ci95_lo": base["ci95_lo"] if base else np.nan,
           "delta_ci95_hi": base["ci95_hi"] if base else np.nan, "p_better_baseline": base["p_better"] if base else np.nan,
           "delta_champion": m["score"] - ch_score, "runtime_min": runtime_s / 60, "peak_gpu_gb": peak,
           "n_train": n_train, "n_val": m["n"], "empty": m["empty"], "truncated": trunc,
           "status": "EVALUATED", "time": time.strftime("%Y-%m-%d %H:%M:%S")}
    for g in ("A", "B"):                                   # per-class score columns for the trial table
        idx = [i for i, c in enumerate(VAL_CLASS) if c == g]
        row[f"score_{g}"] = official_metric([VAL_GT[i] for i in idx], [hyps[i] for i in idx])["score"] if idx else np.nan
    row.update({"lr": CFG.LEARNING_RATE, "aug_profile": CFG.AUG_PROFILE, "class_hint": CFG.CLASS_HINT,
                "class_aug_copies": json.dumps(CFG.CLASS_AUG_COPIES)})
    if extra: row.update(extra)
    prev = pd.read_csv(TRIAL_RESULTS) if os.path.exists(TRIAL_RESULTS) else pd.DataFrame()
    pd.concat([prev, pd.DataFrame([row])], ignore_index=True).to_csv(TRIAL_RESULTS, index=False)
    with open(os.path.join(TRIAL_CFG_DIR, f"{trial}.json"), "w") as f:
        json.dump({"trial": trial, "session": TRIAL_ID, "description": desc, "model": model, "variant": variant,
                   "decode": CFG.DECODE_VARIANTS.get(variant), "cfg": {k: (list(v) if isinstance(v, (set, tuple)) else v)
                   for k, v in TRIAL_CONFIG.items()}}, f, indent=1, default=str)
    return m

In [ ]:
# =========================================================
# Cell 12 — CLASS-WISE: for each base model, train ONE fresh LoRA per class on that class only (with its own
#           prompt), then decode every line with the adapter of its class. One base model resident at a time.
# =========================================================
import torch.nn as _nn

CLASS_KEYS = ["A1", "A2", "A3", "B"] if CFG.CLASS_SCHEME == "subclass" else ["A", "B"]

def class_key(grp):
    return grp if CFG.CLASS_SCHEME == "subclass" else grp[0]

def ocr_prompt(grp=None):
    """overrides Cell 5: each class model has its own prompt, identical in training and inference."""
    return CFG.CLASS_PROMPTS[class_key(grp)] if grp else CFG.OCR_PROMPT

for _k in CLASS_KEYS:
    assert _k in CFG.CLASS_PROMPTS, f"CLASS_PROMPTS has no prompt for class {_k}"
    print(f"[prompt {_k}] {CFG.CLASS_PROMPTS[_k]}")
_cnt = collections.Counter(class_key(s["grp"]) for s in core_samples + pseudo_samples)
print("training lines per class model:", {k: _cnt.get(k, 0) for k in CLASS_KEYS},
      "| val lines per class:", dict(collections.Counter(class_key(g) for g in VAL_GROUP)))

def _vision_linear_names(model):
    return [n for n, m in model.named_modules()
            if isinstance(m, _nn.Linear) and any(t in n.lower() for t in ("visual", "vision", "merger"))]

def _hp(mname, key):
    hp = {k: getattr(CFG, k) for k in ("LEARNING_RATE", "NUM_TRAIN_EPOCHS", "R", "LORA_ALPHA", "AUG_COPIES",
                                        "PER_DEVICE_TRAIN_BATCH_SIZE", "GRADIENT_ACCUMULATION_STEPS")}
    hp.update(CFG.MODEL_OVERRIDES.get(model_size_tag(mname), {}))
    hp.update(CFG.CLASS_HP.get(key, {}))
    return hp

def _rows_for(samples, n_aug, model_num, salt):
    rows = [{"image": s["image"], "text": s["text"], "grp": s["grp"], "aug": False, "seed": -1} for s in samples]
    if CFG.USE_AUGMENT:
        for _j, s in enumerate(samples):
            for _c in range(int(n_aug)):
                rows.append({"image": s["image"], "text": s["text"], "grp": s["grp"], "aug": True,
                             "seed": int((CFG.SEED + 1) * 2000003 + CFG.FOLD * 1000003 + model_num + _c * 9973 + _j
                                         + salt * 7777777) & 0x7fffffff})
    return rows

FOLD_TAG = f"fold{CFG.FOLD}" if CFG.RUN_MODE == "fold0" else "full"

def _find_class_adapter(mname, key):
    names = [model_short_name(m) for m in [mname]] + [mname]
    roots = ([CFG.ADAPTERS_DIR] if CFG.ADAPTERS_DIR else []) + [CFG.FINAL_DIR] + \
            (sorted(_glob("/kaggle/input/*")) + sorted(_glob("/kaggle/input/*/*")) if not CFG.ADAPTERS_DIR else [])
    for r in roots:
        for nm in names:
            p = os.path.join(r, nm, FOLD_TAG, f"class_{key}")
            if os.path.isfile(os.path.join(p, "adapter_config.json")):
                return p
    for r in roots:
        for cfgp in _glob(os.path.join(r, "**", f"class_{key}", "adapter_config.json"), recursive=True):
            parts = os.path.dirname(cfgp).replace("\\", "/").split("/")
            if len(parts) >= 3 and parts[-2] == FOLD_TAG and parts[-3] == mname:
                return os.path.dirname(cfgp)
    return None

BASE_HYPS = None
if CFG.RUN_MODE == "fold0" and CFG.BASELINE_PRED_CSV and os.path.exists(CFG.BASELINE_PRED_CSV):
    _b = pd.read_csv(CFG.BASELINE_PRED_CSV).set_index("ID")
    BASE_HYPS = [norm_text(_b.at[i, "pred"]) if i in _b.index else "" for i in VAL_IDS]
    print(f"baseline predictions loaded for {sum(i in _b.index for i in VAL_IDS)}/{len(VAL_IDS)} val lines "
          f"-> score {official_metric(VAL_GT, BASE_HYPS)['score']:.5f}")

dtype = torch.bfloat16 if (CFG.USE_BF16 and torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else (
        torch.float16 if torch.cuda.is_available() else torch.float32)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
VAL_NBEST, TEST_NBEST, UNL_NBEST, MODEL_ORDER = {}, {}, {}, []
TRAIN_SIZE = len(core_samples)
_EMPTY = [("", float("-inf"), 0, False)]

def decode_split(model, proc, paths, groups, var, desc):
    """every line decoded by the adapter of its own class (batched per class)."""
    out = [None] * len(paths)
    for key in CLASS_KEYS:
        idx = [i for i, g in enumerate(groups) if class_key(g) == key]
        if not idx:
            continue
        model.set_adapter(f"class_{key}")
        sub = transcribe(model, proc, [paths[i] for i in idx], var, desc=f"{desc}/{key}")
        for j, i in enumerate(idx):
            out[i] = sub[j]
    return [o if o is not None else list(_EMPTY) for o in out]

for model_num, msrc in enumerate(CFG.MODEL_PATHS):
    if (time.time() - SESSION_T0) / 3600 > CFG.TIME_BUDGET_H:
        print(f"[time] budget {CFG.TIME_BUDGET_H}h reached — skipping {msrc}"); continue
    mname = model_short_name(msrc); MODEL_ORDER.append(mname)
    print(f"\n=========== BASE MODEL {mname} ({CFG.RUN_MODE}, {FOLD_TAG}) — class models {CLASS_KEYS} ===========")
    t_model = time.time()
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
    proc = load_vlm_processor(msrc, max_pixels=CFG.MAX_PIXELS, min_pixels=CFG.MIN_PIXELS)
    base = load_vlm_model(msrc, dtype); base.to(DEVICE); base.config.use_cache = False
    grad_ckpt = any(s in msrc.lower() for s in CFG.GRAD_CKPT_MODELS)
    if CFG.DO_TRAIN and grad_ckpt:
        base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    base.enable_input_require_grads()
    TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"] + \
              (_vision_linear_names(base) if CFG.LORA_VISION else [])     # computed on the BASE, before wrapping
    ad_dir = os.path.join(CFG.FINAL_DIR, mname, FOLD_TAG)
    model = None
    for key in CLASS_KEYS:
        name = f"class_{key}"
        hp = _hp(mname, key)
        own = [s for s in core_samples + pseudo_samples if class_key(s["grp"]) == key]
        if CFG.DO_TRAIN:
            rows = _rows_for(own, hp["AUG_COPIES"], model_num, salt=CLASS_KEYS.index(key) + 1)
            if CFG.CLASS_REPLAY > 0:
                others = [s for s in core_samples + pseudo_samples if class_key(s["grp"]) != key]
                n_rep = int(round(CFG.CLASS_REPLAY / (1 - CFG.CLASS_REPLAY) * len(rows)))
                pick = np.random.RandomState(CFG.SEED + 13 + model_num).choice(len(others), min(n_rep, len(others)), replace=False)
                rows += [{"image": others[i]["image"], "text": others[i]["text"], "grp": others[i]["grp"], "aug": True,
                          "seed": int(CFG.SEED * 31 + i) & 0x7fffffff} for i in pick]
            lcfg = LoraConfig(r=hp["R"], lora_alpha=hp["LORA_ALPHA"], target_modules=TARGETS,
                              lora_dropout=CFG.LORA_DROPOUT, bias="none", task_type="CAUSAL_LM")
            if model is None:
                model = get_peft_model(base, lcfg, adapter_name=name)
            else:
                model.add_adapter(name, lcfg)
            model.set_adapter(name)
            n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
            print(f"[{mname}] class model {name}: {len(own)} lines -> {len(rows)} rows "
                  f"(replay {CFG.CLASS_REPLAY:.0%}) | hp {hp} | trainable {n_tr/1e6:.1f}M")
            args = TrainingArguments(
                output_dir=os.path.join(CFG.OUTPUT_DIR, f"{mname}_{name}"),
                per_device_train_batch_size=hp["PER_DEVICE_TRAIN_BATCH_SIZE"],
                gradient_accumulation_steps=hp["GRADIENT_ACCUMULATION_STEPS"],
                learning_rate=hp["LEARNING_RATE"], num_train_epochs=hp["NUM_TRAIN_EPOCHS"],
                lr_scheduler_type=CFG.LR_SCHEDULER, warmup_ratio=CFG.WARMUP_RATIO,
                bf16=(dtype == torch.bfloat16), fp16=(dtype == torch.float16), logging_steps=10,
                eval_strategy="no", save_strategy="no", remove_unused_columns=False,
                dataloader_num_workers=CFG.DATALOADER_NUM_WORKERS, dataloader_pin_memory=torch.cuda.is_available(),
                report_to="none", seed=CFG.SEED)
            trainer = Trainer(model=model, args=args, train_dataset=Dataset.from_list(rows),
                              data_collator=lambda x, _p=proc: collate_fn(x, _p))
            t0 = time.time(); trainer.train()
            print(f"[{mname}] {name} trained in {(time.time()-t0)/60:.1f} min | label-mask failures so far: {MASK_FAIL_COUNT}")
            try:
                pd.DataFrame(trainer.state.log_history).to_csv(
                    os.path.join(CFG.WORK_DIR, f"log_history__{mname}__{name}.csv"), index=False)
            except Exception: pass
            if CFG.KEEP_ADAPTERS:                  # PEFT writes non-default adapters to <ad_dir>/<name>/
                os.makedirs(ad_dir, exist_ok=True)
                model.save_pretrained(ad_dir, selected_adapters=[name]); proc.save_pretrained(ad_dir)
            del trainer; gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()
            shutil.rmtree(os.path.join(CFG.OUTPUT_DIR, f"{mname}_{name}"), ignore_errors=True)
        else:
            p = _find_class_adapter(mname, key)
            assert p is not None, f"no saved adapter class_{key} for {mname} ({FOLD_TAG}) under ADAPTERS_DIR={CFG.ADAPTERS_DIR!r}"
            if model is None:
                model = PeftModel.from_pretrained(base, p, adapter_name=name)
            else:
                model.load_adapter(p, adapter_name=name)
            print(f"[{mname}] class model {name} <- {p}")
    model.eval(); model.config.use_cache = True
    try: model.gradient_checkpointing_disable()
    except Exception: pass
    t_train = time.time() - t_model

    if CFG.DO_INFER and CFG.RUN_MODE == "fold0":
        scores = {}
        for vname, var in CFG.DECODE_VARIANTS.items():
            t0 = time.time()
            nb = decode_split(model, proc, VAL_PATHS, VAL_GROUP, var, f"val/{mname}/{vname}")
            VAL_NBEST[(mname, vname)] = nb
            top1 = [x[0][0] for x in nb]
            trial = f"{TRIAL_ID}__{mname}__{vname}"
            save_predictions(trial, VAL_IDS, VAL_GT, nb)
            m = report_trial(trial, TRIAL_DESCRIPTION, f"{mname} (class models {'/'.join(CLASS_KEYS)})", vname, top1,
                             t_train + time.time() - t0, TRAIN_SIZE, sum(x[0][3] for x in nb), baseline_hyps=BASE_HYPS,
                             extra={"class_scheme": CFG.CLASS_SCHEME, "class_replay": CFG.CLASS_REPLAY})
            scores[vname] = m["score"]
            if int(var.get("n_return", 1)) > 1:
                mbr = [mbr_select([t for t, *_ in x], nbest_posteriors([s for _, s, *_ in x], CFG.MBR_TEMPERATURE)) for x in nb]
                report_trial(f"{trial}_mbr", TRIAL_DESCRIPTION, f"{mname} (class models)", f"{vname}+mbr", mbr, 0.0,
                             TRAIN_SIZE, baseline_hyps=BASE_HYPS, extra={"class_scheme": CFG.CLASS_SCHEME})
                scores[f"{vname}+mbr"] = official_metric(VAL_GT, mbr)["score"]
                save_predictions(f"{trial}_mbr", VAL_IDS, VAL_GT, nb, chosen=mbr)
        best_var = max((v for v in scores if "+mbr" not in v), key=scores.get)
        print(f"[{mname}] best decoding on val: {best_var} | " + ", ".join(f"{k}={v:.5f}" for k, v in scores.items()))
        test_vars = {best_var} | {v for v, c in CFG.DECODE_VARIANTS.items() if int(c.get("n_return", 1)) > 1}
    else:
        test_vars = {CFG.CHAMPION_DECODE}

    if CFG.DO_INFER and CFG.PREDICT_TEST:
        for vname in sorted(test_vars):
            nb = decode_split(model, proc, TEST_PATHS, TEST_GROUP, CFG.DECODE_VARIANTS[vname], f"test/{mname}/{vname}")
            TEST_NBEST[(mname, vname)] = nb
            save_predictions(f"{TRIAL_ID}__{mname}__{vname}", TEST_IDS, None, nb, split="test")
    if CFG.DO_INFER and CFG.PREDICT_UNLISTED and UNL_IDS:
        for vname in sorted(v for v in test_vars if int(CFG.DECODE_VARIANTS[v].get("n_return", 1)) > 1):
            nb = decode_split(model, proc, UNL_PATHS, UNL_GROUP, CFG.DECODE_VARIANTS[vname], f"unlisted/{mname}/{vname}")
            UNL_NBEST[(mname, vname)] = nb
            save_predictions(f"{TRIAL_ID}__{mname}__{vname}", UNL_IDS, None, nb, split="unlisted")
    free_cuda("model", "base"); del proc; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    print(f"[{mname}] done in {(time.time()-t_model)/60:.1f} min")

In [ ]:
# =========================================================
# Cell 13 — candidates (per base model and across base models), champion, submission, pseudo-labels
# =========================================================
def _pool(sys_nb):
    cands, w = [], []
    for x in sys_nb:
        p = nbest_posteriors([s for _, s, *_ in x], CFG.MBR_TEMPERATURE)
        cands += [t for t, *_ in x]; w += list(p / len(sys_nb))
    return cands, w

def _ensemble(nb_by_model, method):
    out = []
    for i in range(len(nb_by_model[0])):
        sys_nb = [nb[i] for nb in nb_by_model]
        out.append(mbr_select(*_pool(sys_nb)) if method in ("mbr_nbest", "single_mbr")
                   else mbr_select([x[0][0] for x in sys_nb]))
    return out

_top1 = lambda xs: [x[0][0] for x in xs]
CANDIDATES = {}
for (mn, vn), nb in VAL_NBEST.items():
    CANDIDATES[f"{mn}__{vn}"] = (_top1(nb), _top1(TEST_NBEST[(mn, vn)]) if (mn, vn) in TEST_NBEST else None)
    if int(CFG.DECODE_VARIANTS[vn].get("n_return", 1)) > 1:
        CANDIDATES[f"{mn}__{vn}+mbr"] = (_ensemble([nb], "single_mbr"),
                                         _ensemble([TEST_NBEST[(mn, vn)]], "single_mbr") if (mn, vn) in TEST_NBEST else None)
if CFG.RUN_MODE == "fold0" and len(MODEL_ORDER) >= 2:
    for vname in CFG.DECODE_VARIANTS:
        systems = [m for m in MODEL_ORDER if (m, vname) in VAL_NBEST]
        if len(systems) < 2: continue
        for method in CFG.ENSEMBLE_METHODS:
            if method == "mbr_nbest" and int(CFG.DECODE_VARIANTS[vname].get("n_return", 1)) < 2: continue
            vh = _ensemble([VAL_NBEST[(m, vname)] for m in systems], method)
            th = _ensemble([TEST_NBEST[(m, vname)] for m in systems], method) if all((m, vname) in TEST_NBEST for m in systems) else None
            name = f"ENS[{'+'.join(model_size_tag(s_) for s_ in systems)}]__{vname}__{method}"
            CANDIDATES[name] = (vh, th)
            report_trial(f"{TRIAL_ID}__{name}", TRIAL_DESCRIPTION, name, f"{vname}/{method}", vh, 0.0, TRAIN_SIZE,
                         baseline_hyps=BASE_HYPS, extra={"class_scheme": CFG.CLASS_SCHEME})
            pd.DataFrame({"ID": VAL_IDS, "GroundTruth": VAL_GT, "pred": vh}).to_csv(
                os.path.join(TRIAL_PRED_DIR, f"{TRIAL_ID}__{name}__val.csv"), index=False)

if CFG.RUN_MODE == "fold0":
    ranked = sorted(((official_metric(VAL_GT, v)["score"], k) for k, (v, t) in CANDIDATES.items() if v is not None), reverse=True)
    print("\nVALIDATION RANKING (official score):")
    for s, k in ranked[:12]:
        print(f"  {s:.5f}  {k}{'' if CANDIDATES[k][1] is not None else '   (no test preds)'}")
    best = next((k for s, k in ranked if CANDIDATES[k][1] is not None), None)
else:
    vn = CFG.CHAMPION_DECODE
    systems = [m for m in MODEL_ORDER if (m, vn) in TEST_NBEST]
    nbr = int(CFG.DECODE_VARIANTS[vn].get("n_return", 1))
    if len(systems) >= 2 and CFG.CHAMPION_ENSEMBLE:
        best = f"ENS[{'+'.join(model_size_tag(s_) for s_ in systems)}]__{vn}__{CFG.CHAMPION_ENSEMBLE}"
        CANDIDATES[best] = (None, _ensemble([TEST_NBEST[(m, vn)] for m in systems], CFG.CHAMPION_ENSEMBLE))
    elif systems:
        best = f"{systems[0]}__{vn}" + ("+mbr" if (CFG.CHAMPION_ENSEMBLE and nbr > 1) else "")
        nb0 = TEST_NBEST[(systems[0], vn)]
        CANDIDATES[best] = (None, _ensemble([nb0], "single_mbr") if (CFG.CHAMPION_ENSEMBLE and nbr > 1) else _top1(nb0))
    else:
        best = None

if best is not None and CANDIDATES[best][1] is not None:
    sub = pd.DataFrame({"ID": TEST_IDS, "Target": [norm_text(t) or "the" for t in CANDIDATES[best][1]]})
    sub.to_csv(CFG.SUBMISSION_CSV, index=False)
    print(f"\nsubmission.csv <- {best} | rows={len(sub)} | test lines per class model:",
          dict(collections.Counter(class_key(g) for g in TEST_GROUP)))
    with open(os.path.join(CFG.WORK_DIR, "submission_source.json"), "w") as f:
        json.dump({"trial_session": TRIAL_ID, "source": best, "class_scheme": CFG.CLASS_SCHEME}, f)

# pseudo-labels (test + unlisted): MBR over every base model's class-routed N-best; risk = expected official cost
_pl_rows = []
for split_ids, store in ((TEST_IDS, TEST_NBEST), (UNL_IDS, UNL_NBEST)):
    keys = [k for k in store if int(CFG.DECODE_VARIANTS[k[1]].get("n_return", 1)) > 1]
    for i, rid in enumerate(split_ids if keys else []):
        txt, risk = mbr_select(*_pool([store[k][i] for k in keys]), return_risk=True)
        _pl_rows.append({"ID": rid, "Target": txt, "risk": risk, "n_systems": len(keys)})
if _pl_rows:
    pd.DataFrame(_pl_rows).to_csv(os.path.join(CFG.WORK_DIR, "pseudo_labels.csv"), index=False)
    print(f"pseudo_labels.csv: {len(_pl_rows)} lines (refine with work/08_build_pseudo_labels.py)")

if os.path.exists(TRIAL_RESULTS):
    res = pd.read_csv(TRIAL_RESULTS)
    sess = res.session == TRIAL_ID
    if sess.any():
        bi = res[sess].score.idxmax()
        res.loc[sess & (res.index != bi) & (res.status == "EVALUATED"), "status"] = "REJECTED"
        res.loc[bi, "status"] = "PROMOTED"
        res.to_csv(TRIAL_RESULTS, index=False)
        print("\nTRIAL TABLE (this session):")
        print(res[sess][["trial", "variant", "score", "score_A", "score_B", "word_edits_per_line", "char_edits_per_line",
                         "delta_baseline", "p_better_baseline", "status"]].to_string(index=False))

In [ ]:
# =========================================================
# Cell 14 — per-class comparison and error forensics of the best class-wise system
# =========================================================
if CFG.RUN_MODE == "fold0" and CANDIDATES:
    vals = {k: v for k, (v, t) in CANDIDATES.items() if v is not None}
    champ = max(vals, key=lambda k: official_metric(VAL_GT, vals[k])["score"])
    print("CHAMPION:", champ)
    print("per class :", score_by_group(VAL_GT, vals[champ], VAL_CLASS))
    print("per group :", score_by_group(VAL_GT, vals[champ], VAL_GROUP))
    if BASE_HYPS is not None:
        print("\nCLASS-WISE vs BASELINE (shared model), paired bootstrap per class:")
        for c in sorted(set(VAL_CLASS)) + sorted(set(VAL_GROUP) - set(VAL_CLASS)):
            idx = [i for i, (a, g) in enumerate(zip(VAL_CLASS, VAL_GROUP)) if c in (a, g)]
            d = paired_delta([VAL_GT[i] for i in idx], [BASE_HYPS[i] for i in idx], [vals[champ][i] for i in idx])
            print(f"  {c:3s} n={len(idx):4d}  delta={d['delta']:+.5f}  CI95 [{d['ci95_lo']:+.5f}, {d['ci95_hi']:+.5f}]  "
                  f"P(class-wise better)={d['p_better']:.3f}  word/char edits saved={d['word_edits_saved']}/{d['char_edits_saved']}")
        d = paired_delta(VAL_GT, BASE_HYPS, vals[champ])
        print(f"  ALL n={len(VAL_GT)}  delta={d['delta']:+.5f}  CI95 [{d['ci95_lo']:+.5f}, {d['ci95_hi']:+.5f}]  P={d['p_better']:.3f}")
    fx = error_forensics(VAL_GT, vals[champ])
    for k, v in fx.items():
        print(f"  {k}: {v}")

### Suggested runs
| Run | CFG | Why |
|---|---|---|
| CLS_001 | defaults: `CLASS_SCHEME="class"` (A, B), 8B, no replay | Two separate era models with era-specific prompts |
| CLS_002 | `CLASS_SCHEME="subclass"` | Four separate models (A1, A2, A3, B). A2 has only ~300 training lines |
| CLS_003 | `CLASS_REPLAY=0.15` | Mostly separate models that also see 15 % other-class lines |
| CLS_004 | `CLASS_HP={"A2": {"NUM_TRAIN_EPOCHS": 2}}` | More passes for the smallest class |
| FINAL | `RUN_MODE="full"`, same scheme and prompts, `CHAMPION_DECODE` / `CHAMPION_ENSEMBLE` | Retrain on all 4,093 lines |

Set `BASELINE_PRED_CSV` to a shared-model validation file from `barbados-2-enhanced.ipynb` (e.g. `trial_predictions/EXP_002__qwen_3_vl_transformers_8b_instruct_1__D0_orig__val.csv`, measured 0.90947) to get the per-class verdict. Keep the class-wise models only where the difference is positive with P ≥ 0.9.